In [1]:
import pandas as pd
from pathlib import Path
from statsmodels.multivariate.manova import MANOVA

## 1. ĐỌC DỮ LIỆU

In [2]:
print("BƯỚC 1: ĐỌC DỮ LIỆU SẠCH")

df = pd.read_excel(Path("..") / "data" / "processed" / "clean_data.xlsx")

print("Kích thước dữ liệu:", df.shape)

print("\nCác biến:")
print(df.columns.tolist())

BƯỚC 1: ĐỌC DỮ LIỆU SẠCH
Kích thước dữ liệu: (720, 7)

Các biến:
['Fragrance', 'Flavor', 'Aftertaste', 'Acidity', 'Body', 'Moisture', 'Origin']


## 2. KHAI BÁO CÁC BIẾN

In [3]:
dependent_vars = [
    "Fragrance",
    "Flavor",
    "Aftertaste",
    "Acidity",
    "Body",
    "Moisture"
]

group_var = "Origin"

## 3. KIỂM TRA DỮ LIỆU

In [4]:
print("BƯỚC 2: KIỂM TRA DỮ LIỆU")

# Kiểm tra missing
missing = df[dependent_vars + [group_var]].isnull().sum()

print("\nMissing:")
print(missing)

# Kiểm tra nhóm
print("\nSố lượng mẫu theo Origin:")
print(df[group_var].value_counts())

# Kiểm tra đúng 4 nhóm
expected_groups = [
    "Mexico",
    "Colombia",
    "Guatemala",
    "Brazil"
]

actual_groups = sorted(df[group_var].dropna().unique())

print("\nCác nhóm thực tế:")
print(actual_groups)

if sorted(expected_groups) != actual_groups:
    print("\n⚠ Cảnh báo: Nhóm Origin không đúng như dự kiến.")
else:
    print("\n✓ Dữ liệu có đúng 4 nhóm Origin.")



BƯỚC 2: KIỂM TRA DỮ LIỆU

Missing:
Fragrance     0
Flavor        0
Aftertaste    0
Acidity       0
Body          0
Moisture      0
Origin        0
dtype: int64

Số lượng mẫu theo Origin:
Origin
Mexico       236
Colombia     180
Guatemala    175
Brazil       129
Name: count, dtype: int64

Các nhóm thực tế:
['Brazil', 'Colombia', 'Guatemala', 'Mexico']

✓ Dữ liệu có đúng 4 nhóm Origin.


## 4. THỐNG KÊ MÔ TẢ

In [5]:
print("BƯỚC 3: THỐNG KÊ MÔ TẢ")

# Trung bình theo nhóm
group_mean = (
    df
    .groupby(group_var)[dependent_vars]
    .mean()
)

print("\nTrung bình của 6 biến theo Origin:")
print(group_mean)


# Độ lệch chuẩn theo nhóm
group_std = (
    df
    .groupby(group_var)[dependent_vars]
    .std()
)

print("\nĐộ lệch chuẩn:")
print(group_std)


# Số mẫu theo nhóm
group_n = (
    df
    .groupby(group_var)
    .size()
    .reset_index(name="N")
)

print("\nSố mẫu:")
print(group_n)

BƯỚC 3: THỐNG KÊ MÔ TẢ

Trung bình của 6 biến theo Origin:
           Fragrance    Flavor  Aftertaste   Acidity      Body  Moisture
Origin                                                                  
Brazil      7.553023  7.570155    7.450775  7.505891  7.538450  0.083721
Colombia    7.666722  7.595944    7.522333  7.571167  7.621056  0.062444
Guatemala   7.571086  7.521029    7.294057  7.604971  7.483543  0.101886
Mexico      7.457585  7.379025    7.228051  7.428347  7.380466  0.113602

Độ lệch chuẩn:
           Fragrance    Flavor  Aftertaste   Acidity      Body  Moisture
Origin                                                                  
Brazil      0.322283  0.279385    0.266214  0.251295  0.248883  0.048494
Colombia    0.189813  0.177527    0.210164  0.226507  0.206824  0.057272
Guatemala   0.269050  0.319162    0.348343  0.275758  0.279427  0.025806
Mexico      0.274343  0.328595    0.315775  0.266398  0.212731  0.032093

Số mẫu:
      Origin    N
0     Brazil  129
1   

## 5. XÂY DỰNG CÔNG THỨC MANOVA

In [6]:
# Dạng:
# Y1 + Y2 + Y3 + ... + Y6 ~ Origin
formula = (
    "Fragrance + Flavor + Aftertaste + "
    "Acidity + Body + Moisture ~ Origin"
)

print("BƯỚC 4: CHẠY MANOVA")

print("\nCông thức:")
print(formula)

BƯỚC 4: CHẠY MANOVA

Công thức:
Fragrance + Flavor + Aftertaste + Acidity + Body + Moisture ~ Origin


## 6. CHẠY MANOVA

In [7]:
manova = MANOVA.from_formula(
    formula,
    data=df
)

manova_result = manova.mv_test()

print("\nKẾT QUẢ MANOVA:")
print(manova_result)


KẾT QUẢ MANOVA:
                    Multivariate linear model
                                                                  
------------------------------------------------------------------
       Intercept         Value   Num DF  Den DF   F Value   Pr > F
------------------------------------------------------------------
          Wilks' lambda   0.0042 6.0000 711.0000 28358.4774 0.0000
         Pillai's trace   0.9958 6.0000 711.0000 28358.4774 0.0000
 Hotelling-Lawley trace 239.3120 6.0000 711.0000 28358.4774 0.0000
    Roy's greatest root 239.3120 6.0000 711.0000 28358.4774 0.0000
------------------------------------------------------------------
                                                                  
------------------------------------------------------------------
            Origin         Value   Num DF   Den DF  F Value Pr > F
------------------------------------------------------------------
             Wilks' lambda 0.5961 18.0000 2011.4970 22.4271 0.0000

## 7. TRÍCH XUẤT KẾT QUẢ MANOVA

In [8]:
print("\n" + "=" * 70)
print("BƯỚC 5: KẾT QUẢ CÁC KIỂM ĐỊNH ĐA BIẾN")
print("=" * 70)

# Lấy kết quả Origin
origin_result = manova_result.results["Origin"]

stat_table = origin_result["stat"].copy()

print("\nBảng thống kê:")
print(stat_table)


BƯỚC 5: KẾT QUẢ CÁC KIỂM ĐỊNH ĐA BIẾN

Bảng thống kê:
                           Value Num DF       Den DF    F Value Pr > F
Wilks' lambda           0.596122     18  2011.496967  22.427123    0.0
Pillai's trace          0.446667   18.0       2139.0  20.788107    0.0
Hotelling-Lawley trace  0.607239     18  1416.011252  23.952287    0.0
Roy's greatest root     0.466878      6          713  55.480646    0.0


## 8. GIẢI THÍCH CÁC THỐNG KÊ

In [9]:
print("BƯỚC 6: GIẢI THÍCH KẾT QUẢ")

print(
    "\nCác kiểm định được báo cáo gồm:"
)

print("1. Wilks' Lambda")
print("2. Pillai's Trace")
print("3. Hotelling-Lawley Trace")
print("4. Roy's Greatest Root")

BƯỚC 6: GIẢI THÍCH KẾT QUẢ

Các kiểm định được báo cáo gồm:
1. Wilks' Lambda
2. Pillai's Trace
3. Hotelling-Lawley Trace
4. Roy's Greatest Root


## 9. LẤY P-VALUE

In [10]:
# Tên dòng trong statsmodels
# thường gồm:
# Wilks' lambda
# Pillai's trace
# Hotelling-Lawley trace
# Roy's greatest root
p_values = {}

for test_name in stat_table.index:
    p_value = stat_table.loc[test_name, "Pr > F"]
    p_values[test_name] = p_value


## 10. KẾT LUẬN MANOVA

In [11]:
print("KẾT LUẬN MANOVA")

alpha = 0.05

print("\nMức ý nghĩa alpha =", alpha)

# Sử dụng Wilks' Lambda làm chỉ số chính
wilks_p = stat_table.loc[
    "Wilks' lambda",
    "Pr > F"
]

print("\nWilks' Lambda p-value =", wilks_p)

if wilks_p < alpha:

    print(
        "\nKết luận:"
    )

    print(
        "Bác bỏ H0 ở mức ý nghĩa 5%."
    )

    print(
        "Có bằng chứng thống kê cho thấy "
        "Origin có ảnh hưởng đến tập hợp "
        "6 biến phụ thuộc: Fragrance, Flavor, "
        "Aftertaste, Acidity, Body và Moisture."
    )

else:

    print(
        "\nKết luận:"
    )

    print(
        "Chưa đủ bằng chứng để bác bỏ H0 ở mức ý nghĩa 5%."
    )

    print(
        "Chưa kết luận được rằng Origin tạo ra "
        "sự khác biệt có ý nghĩa thống kê trên "
        "tập hợp 6 biến phụ thuộc."
    )


KẾT LUẬN MANOVA

Mức ý nghĩa alpha = 0.05

Wilks' Lambda p-value = 2.210310704526228e-67

Kết luận:
Bác bỏ H0 ở mức ý nghĩa 5%.
Có bằng chứng thống kê cho thấy Origin có ảnh hưởng đến tập hợp 6 biến phụ thuộc: Fragrance, Flavor, Aftertaste, Acidity, Body và Moisture.


## 11. LẤY KẾT QUẢ CỦA TỪNG TEST

In [12]:
summary_results = []

for test_name in stat_table.index:

    row = {
        "Test": test_name,
        "Value": stat_table.loc[test_name, "Value"],
        "Num DF": stat_table.loc[test_name, "Num DF"],
        "Den DF": stat_table.loc[test_name, "Den DF"],
        "F Value": stat_table.loc[test_name, "F Value"],
        "p-value": stat_table.loc[test_name, "Pr > F"]
    }

    summary_results.append(row)

manova_summary = pd.DataFrame(summary_results)

## 12. ĐÁNH DẤU Ý NGHĨA THỐNG KÊ

In [13]:
manova_summary["Significant_at_0.05"] = (
    manova_summary["p-value"] < 0.05
)

print("\nBảng tổng hợp:")
print(manova_summary)


Bảng tổng hợp:
                     Test     Value  Num DF       Den DF    F Value  \
0           Wilks' lambda  0.596122    18.0  2011.496967  22.427123   
1          Pillai's trace  0.446667    18.0  2139.000000  20.788107   
2  Hotelling-Lawley trace  0.607239    18.0  1416.011252  23.952287   
3     Roy's greatest root  0.466878     6.0   713.000000  55.480646   

        p-value  Significant_at_0.05  
0  2.210311e-67                 True  
1  1.460384e-62                 True  
2  2.809722e-69                 True  
3  3.148612e-56                 True  


## 13. XUẤT KẾT QUẢ EXCEL

In [14]:
RESULT_FILE = Path("..") / "data" / "processed" / "manova_results.xlsx"

# Create the directory if it does not exist
Path(RESULT_FILE).parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(
    RESULT_FILE,
    engine="openpyxl"
) as writer:

    # Thống kê trung bình
    group_mean.to_excel(
        writer,
        sheet_name="Group_Mean"
    )

    # Độ lệch chuẩn
    group_std.to_excel(
        writer,
        sheet_name="Group_SD"
    )

    # Số mẫu
    group_n.to_excel(
        writer,
        sheet_name="Group_N",
        index=False
    )

    # Kết quả MANOVA
    manova_summary.to_excel(
        writer,
        sheet_name="MANOVA",
        index=False
    )

## 14. HOÀN THÀNH

In [15]:
print("HOÀN THÀNH MANOVA")
print("\nFile kết quả:")
print(RESULT_FILE)

HOÀN THÀNH MANOVA

File kết quả:
..\data\processed\manova_results.xlsx
